# 🐄 ĐỀ TÀI QUÈ CHÂN – SO SÁNH MÔ HÌNH CHƯA CHỈNH SỬA vs ĐÃ CẢI TIẾN

Notebook này chạy trên **Google Colab (CPU)**, tải dữ liệu gốc từ Internet, huấn luyện
và **so sánh trực tiếp**:

| | Cấu hình | Kết quả dự kiến (luận văn) |
|---|---|---|
| ❌ Chưa chỉnh sửa | **M0:** HistGradientBoosting đơn trên đặc trưng tự tham chiếu | AUC 0,892 |
| ❌ Chưa chỉnh sửa | **STACK v3:** ghép xác suất 4 mô hình bằng hồi quy logistic (stacking) | AUC 0,883 |
| ✅ Đã cải tiến | **ENS4:** ensemble **xếp hạng** 4 mô hình (HGB + RF + SVM + LR) | AUC **0,904** |
| ✅ Đã cải tiến (tuỳ chọn) | **ENS8:** thêm MLP, k-NN + 2 mô hình chuẩn-hoá-trong-bò | recall sàng lọc **90,0%** |

Giao thức: **độc lập cá thể** – GroupKFold 10 theo mã bò (mỗi con bò chỉ nằm trọn ở một phía học/chấm).
Thời gian chạy toàn bộ khoảng **5–10 phút** trên Colab CPU.

**Cách dùng:** upload file này lên Colab → Runtime → Run all.

In [ ]:
!pip install -q openpyxl
import time, requests
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier as HGB
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, recall_score
import matplotlib.pyplot as plt
RNG = 42; np.random.seed(RNG)
print("Sẵn sàng.")

In [ ]:
# ---------- TẢI DỮ LIỆU GỐC TỪ ZENODO (bản ghi 18462150, giấy phép CC BY 4.0) ----------
URL = "https://zenodo.org/records/18462150/files/Physiological_and_behavioral_data_2024.xlsx"
r = requests.get(URL, timeout=180); r.raise_for_status()
open("locomotion_data.xlsx", "wb").write(r.content)
print("Đã tải", len(r.content), "bytes")

In [ ]:
# ---------- ĐỌC & LÀM SẠCH (giống hệt pipeline của luận văn) ----------
ld = pd.read_excel("locomotion_data.xlsx", sheet_name="Munka1")
ld = ld.drop(columns=["Szívritmus paraméterek"], errors="ignore")
ld.columns = [x.strip() for x in ld.columns]
sc = [x for x in ld.columns if x.startswith("Mozgáskép")][0]
ld = ld.rename(columns={sc: "score", "Fülszám": "cow", "Vizsgálat időpontja": "date"})
ld["cow"] = ld["cow"].astype(str); ld = ld.dropna(subset=["score"])
ld["y"] = (ld["score"] >= 2).astype(int)      # nhãn: điểm vận động >= 2 là QUÈ
print("Số bản ghi:", len(ld), "| Số bò:", ld['cow'].nunique(), "| Tỷ lệ què: %.1f%%" % (100*ld['y'].mean()))

In [ ]:
# ---------- ĐẶC TRƯNG: thô + hành vi/hệ thống vắt sữa + TỰ THAM CHIẾU (z trong bò) ----------
num = ["Laktáció száma","Tejtermelés (kg) vizsgálat idején","Kondíció",
 "Mean RR  (ms)","STD RR (ms)","Mean HR (1/min)","RMSSD (ms)","NN50 (count)","pNN50 (%)",
 "RR tri index","TINN (ms)","LF (ms2)","HF (ms2)","VLF (%)","LF (%)","HF (%)",
 "LF (n.u.)","HF (n.u.)","LF/HF ratio","SD1 (ms)","SD2 (ms)","SD2/SD1","Mean line length (beats)"]
num = [x for x in num if x in ld.columns]

def base_behavior(s):
    if not isinstance(s, str): return "other"
    if s.startswith("Fekve kérődzés"): return "lr"
    if s.startswith("Állva kérődzés"): return "sr"
    if s.startswith("Fekvés"): return "lying"
    if s.startswith("Étkezés"): return "eat"
    if s.startswith("Állás"): return "stand"
    return "other"
beh = pd.get_dummies(ld["Eseményszótár"].map(base_behavior), prefix="b").astype(float)
ms  = pd.get_dummies(ld["Fejési rendszer kód"].astype(str), prefix="ms").astype(float)

# tự tham chiếu: lệch bao nhiêu độ lệch chuẩn so với mức nền của CHÍNH con bò đó
cow_med = ld.groupby("cow")[num].transform("median")
cow_std = ld.groupby("cow")[num].transform("std").replace(0, np.nan).fillna(1)
dev = (ld[num] - cow_med) / cow_std
dev.columns = ["dev_" + x for x in num]
herd_med = ld.groupby("date")["Tejtermelés (kg) vizsgálat idején"].transform("median")
ld["milk_rel_herd"] = ld["Tejtermelés (kg) vizsgálat idején"] - herd_med

X_dev     = pd.concat([ld[num].astype(float), beh, ms, dev, ld[["milk_rel_herd"]]], axis=1)
X_cownorm = pd.concat([dev, beh, ms, ld[["milk_rel_herd"]]], axis=1)
y, groups = ld["y"].values, ld["cow"].values
print("Đặc trưng X_dev:", X_dev.shape[1], "| X_cownorm:", X_cownorm.shape[1])

---
## GIAO THỨC ĐÁNH GIÁ – ĐỘC LẬP CÁ THỂ
GroupKFold 10: chia 209 bò làm 10 nhóm; mỗi lượt một nhóm bò **chưa từng xuất hiện ở phía học**
được đưa ra chấm. Kết quả đo theo cách này thấp hơn kiểm định thường nhưng trung thực hơn,
vì bò mới ngoài trang trại luôn là bò máy chưa từng gặp.

In [ ]:
# ---------- XÁC SUẤT NGOÀI-FOLD (OOF) CỦA 4 MÔ HÌNH NỀN TRÊN X_dev ----------
t0 = time.time()
folds = list(GroupKFold(10).split(X_dev, y, groups))

def pipe(m, scaled):
    st = [("imp", SimpleImputer(strategy="median"))]
    if scaled: st.append(("sc", StandardScaler()))
    st.append(("clf", m)); return Pipeline(st)

oof = {n: np.zeros(len(y)) for n in ["hgb", "rf", "svm", "lr"]}
for tr, te in folds:
    for name, fn, scaled in [
        ("hgb", lambda: HGB(random_state=RNG, max_iter=300), False),
        ("rf",  lambda: RandomForestClassifier(300, class_weight="balanced", random_state=RNG), False),
        ("svm", lambda: SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RNG), True),
        ("lr",  lambda: LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RNG), True)]:
        m = pipe(fn(), scaled)
        m.fit(X_dev.values[tr], y[tr])
        oof[name][te] = m.predict_proba(X_dev.values[te])[:, 1]
print("OOF 4 mô hình nền xong sau %.0f giây" % (time.time() - t0))

In [ ]:
# ================= MÔ HÌNH CHƯA CHỈNH SỬA =================
def metric(pr, tag):
    d = {"tag": tag, "acc": accuracy_score(y, pr > 0.5), "f1": f1_score(y, pr > 0.5), "auc": roc_auc_score(y, pr)}
    return d

# (1) M0 – HGB đơn (bản v2 trước cải tiến)
m0 = metric(oof["hgb"], "M0 HGB đơn (chưa sửa)")

# (2) STACK v3 – ghép XÁC SUẤT 4 mô hình bằng hồi quy logistic (cách cũ của v3)
oof_stack = np.zeros(len(y))
for tr, te in folds:
    Ztr = np.column_stack([oof[k][tr] for k in oof]); Zte = np.column_stack([oof[k][te] for k in oof])
    meta = LogisticRegression(max_iter=2000).fit(Ztr, y[tr])
    oof_stack[te] = meta.predict_proba(Zte)[:, 1]
stack = metric(oof_stack, "STACK v3 xác suất (chưa sửa)")

for d in (m0, stack):
    print("%-32s acc=%.4f  f1=%.4f  AUC=%.4f" % (d["tag"], d["acc"], d["f1"], d["auc"]))

In [ ]:
# ================= MÔ HÌNH ĐÃ CẢI TIẾN: ENS4 – ENSEMBLE XẾP HẠNG =================
# Thay vì trung bình/stacking XÁC SUẤT (lệch chuẩn định giữa các mô hình),
# đổi mỗi mô hình sang XẾP HẠNG [0,1] rồi lấy trung bình hạng.
def rank_avg(*arrs):
    n = len(arrs[0])
    return np.mean([rankdata(a) / (n + 1) for a in arrs], axis=0)

ens4 = rank_avg(oof["hgb"], oof["rf"], oof["svm"], oof["lr"])
d4 = metric(ens4, "ENS4 xếp hạng 4 mô hình (ĐÃ SỬA)")

# ngưỡng Youden cho điểm vận hành sàng lọc
fpr, tpr, ths = roc_curve(y, ens4); thr = ths[np.argmax(tpr - fpr)]
d4["thr_youden"] = thr
d4["acc_youden"]  = accuracy_score(y, ens4 > thr)
d4["f1_youden"]   = f1_score(y, ens4 > thr)
d4["recall_youden"] = recall_score(y, ens4 > thr)
print("%-32s acc=%.4f  f1=%.4f  AUC=%.4f" % (d4["tag"], d4["acc"], d4["f1"], d4["auc"]))
print("Ngưỡng Youden = %.3f -> acc=%.4f  f1=%.4f  recall bò què=%.1f%%"
      % (thr, d4["acc_youden"], d4["f1_youden"], 100*d4["recall_youden"]))

## (Tuỳ chọn) ENS6 / ENS8 – thêm thành phần đa dạng
Ô dưới huấn luyện thêm 4 mô hình nữa: HGB và RF trên đặc trưng **chuẩn hoá trong bò**,
cộng MLP và k-NN. Nếu muốn chạy nhanh có thể bỏ qua ô này — kết luận chính không thay đổi.

In [ ]:
# ---------- ENS6 / ENS8 (tuỳ chọn) ----------
RUN_EXTRA = True          # đặt False nếu muốn chạy nhanh
res = {"m0": m0, "stack": stack, "ens4": d4}
if RUN_EXTRA:
    t0 = time.time()
    oof2 = {n: np.zeros(len(y)) for n in ["hgb_cn", "rf_cn", "mlp", "knn"]}
    for tr, te in folds:
        for name, X, fn, scaled in [
            ("hgb_cn", X_cownorm, lambda: HGB(random_state=RNG, max_iter=300), False),
            ("rf_cn",  X_cownorm, lambda: RandomForestClassifier(300, class_weight="balanced", random_state=RNG), False),
            ("mlp",    X_dev,     lambda: MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=RNG), True),
            ("knn",    X_dev,     lambda: KNeighborsClassifier(n_neighbors=15), True)]:
            m = pipe(fn(), scaled); m.fit(X.values[tr], y[tr])
            oof2[name][te] = m.predict_proba(X.values[te])[:, 1]
    print("OOF 4 mô hình bổ sung xong sau %.0f giây" % (time.time() - t0))

    for tag, members in [("ens6", ["hgb","rf","svm","lr","hgb_cn","rf_cn"]),
                         ("ens8", ["hgb","rf","svm","lr","hgb_cn","rf_cn","mlp","knn"])]:
        allO = {**oof, **oof2}
        p = rank_avg(*[allO[k] for k in members])
        d = metric(p, "%s xếp hạng %d mô hình (ĐÃ SỬA)" % (tag.upper(), len(members)))
        fpr, tpr, ths = roc_curve(y, p); t = ths[np.argmax(tpr - fpr)]
        d["recall_youden"] = recall_score(y, p > t); d["thr_youden"] = t
        res[tag] = d
        print("%-32s acc=%.4f  f1=%.4f  AUC=%.4f | recall Youden=%.1f%%"
              % (d["tag"], d["acc"], d["f1"], d["auc"], 100*d["recall_youden"]))
else:
    print("Bỏ qua ENS6/ENS8 (RUN_EXTRA = False).")

In [ ]:
# ================= BẢNG SO SÁNH TỔNG HỢP =================
rows = []
for k in ["m0", "stack", "ens4", "ens6", "ens8"]:
    if k not in res: continue
    d = res[k]
    rows.append({"Cấu hình": d["tag"],
                 "Acc (ngưỡng 0,5)": round(d["acc"], 4), "F1 (0,5)": round(d["f1"], 4), "AUC": round(d["auc"], 4),
                 "Recall sàng lọc (Youden)": round(d.get("recall_youden", float("nan")), 4) if "recall_youden" in d else "—"})
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
names = [r["Cấu hình"].split(" (")[0] for r in rows]
axes[0].bar(names, [r["AUC"] for r in rows], color=["#9aa5b1","#9aa5b1","#2e7d32","#43a047","#66bb6a"][:len(rows)])
axes[0].axhline(0.897, color="red", ls="--", lw=1, label="mốc v3 = 0,897")
axes[0].set_ylim(0.85, 0.915); axes[0].set_ylabel("AUC"); axes[0].set_title("AUC: chưa sửa vs đã sửa")
axes[0].legend(); axes[0].tick_params(axis="x", rotation=15)
for tag, p, col in [("M0 HGB", oof["hgb"], "#9aa5b1"), ("STACK v3", oof_stack, "#5c677d"), ("ENS4", ens4, "#2e7d32")]:
    fpr, tpr, _ = roc_curve(y, p)
    axes[1].plot(fpr, tpr, color=col, label="%s (AUC %.3f)" % (tag, roc_auc_score(y, p)))
axes[1].plot([0,1],[0,1],"k:",lw=0.8); axes[1].set_xlabel("Tỷ lệ báo nhầm"); axes[1].set_ylabel("Tỷ lệ bắt đúng bò què")
axes[1].set_title("Đường ROC"); axes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

## ĐỌC KẾT QUẢ
- **STACK v3 (xác suất) thấp hơn cả mô hình đơn** – vì xác suất của SVM/HGB lệch chuẩn định khác nhau,
  tầng logistic lại ít dữ liệu để học trọng số → ghép sai cách còn hại hơn không ghép.
- **ENS4 (xếp hạng) vượt cả hai**: AUC tăng từ 0,892 lên **0,904**, vượt mốc 0,897 của v3.
  Xếp hạng chỉ giữ lại phần thông tin đúng (ai bệnh hơn ai) và bỏ phần nhiễu (mô hình nào tự tin quá đà).
- Ở **điểm vận hành sàng lọc** (ngưỡng Youden), bản 8 mô hình đạt **recall ≈ 90%** —
  10 con bò què chỉ còn sót khoảng 1, tốt hơn 87,45% của phiên bản cũ.
- Số liệu chi tiết trong luận văn: `results/compare_lameness.json` (Mục 4.8).